In [13]:
from pathlib import Path

import pandas as pd

data_path = Path("../data/raw/Earthquake.csv")
df = pd.read_csv(data_path)


df.head()

,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,...,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource
0,2026-11-11T23:00:22.433Z,27.8692,87.8857,10.0,4.6,mb,40.0,161.0,9.216,0.91,...,2026-11-13T22:07:44.819Z,"73 km NNW of Gyalshing, India",earthquake,14.40,1.910,0.093,34.0,reviewed,us,us
1,2026-10-25T07:21:37.857Z,29.6017,80.8002,10.0,4.1,mb,22.0,166.0,7.584,0.99,...,2026-11-07T13:40:10.040Z,"35 km SE of Dārchulā, Nepal",earthquake,9.49,1.890,0.185,8.0,reviewed,us,us
2,2026-10-24T19:34:13.347Z,28.7608,88.4110,10.0,4.3,mb,60.0,149.0,9.558,1.23,...,2026-11-13T23:27:59.040Z,"71 km SW of Rikaze, China",earthquake,10.56,1.900,0.083,41.0,reviewed,us,us
3,2026-10-22T17:11:27.957Z,28.0314,87.3256,10.0,4.5,mb,55.0,159.0,9.685,0.65,...,2026-11-08T16:22:42.040Z,"51 km E of Lobuche, Nepal",earthquake,10.65,1.887,0.081,44.0,reviewed,us,us
4,2026-10-16T19:35:17.741Z,28.6874,87.3652,10.0,4.2,mb,19.0,160.0,10.117,0.71,...,2026-11-07T22:32:37.040Z,"98 km NNE of Lobuche, Nepal",earthquake,14.88,1.920,0.166,11.0,reviewed,us,us


### Initial data summary

- The dataset contains 1,579 rows and 22 columns.
- There are no duplicate rows.
- 8 columns contain missing values.
- The columns with the most missing values are:
    - `nst` (959): Number of seismic stations used to determine the earthquake's location.
    - `horizontalError` (849): Estimated uncertainty in the earthquake's horizontal location, measured in kilometers.
    - `dmin` (838): Angular distance between the earthquake's epicenter and the nearest seismic station, measured in degrees.
- Most missing values are related to measurement quality and uncertainty.
- The dataset contains 1,576 earthquakes and 3 landslides.
- Since the analysis focuses on earthquakes, the 3 landslide records can be removed since we are only looking for earthquake patterns.

In [37]:
# Number of rows and column
print("Data shape", df.shape)

# Null values in each column
print(df.isnull().sum())

# Duplicate rows
print("Num of duplicate rows",df.duplicated().sum())

# Types of event causing earthquake
print(df["type"].value_counts(dropna=False))

df.dtypes


Data shape (1579, 22)
time                 0
latitude             0
longitude            0
depth                0
mag                  0
magType              0
nst                959
gap                530
dmin               838
rms                 97
net                  0
id                   0
updated              0
place                0
type                 0
horizontalError    849
depthError         577
magError           802
magNst             155
status               0
locationSource       0
magSource            0
dtype: int64
Num of duplicate rows 0
type
earthquake    1576
landslide        3
Name: count, dtype: int64


time                object
latitude           float64
longitude          float64
depth              float64
mag                float64
magType             object
nst                float64
gap                float64
dmin               float64
rms                float64
net                 object
id                  object
updated             object
place               object
type                object
horizontalError    float64
depthError         float64
magError           float64
magNst             float64
status              object
locationSource      object
magSource           object
dtype: object

### Keep earthquakes within Nepal’s latitude and longitude ranges

We keep earthquakes inside these ranges because the study focuses on earthquakes in the Nepal area. The ranges form a rectangle, so it can include parts of India and China.

In [21]:
df_nepal = df[
    df["latitude"].between(26 + 22 / 60, 30 + 27 / 60)
    & df["longitude"].between(80 + 4 / 60, 88 + 12 / 60)
    & (df["type"] == "earthquake")
].copy()

print(f"Kept {len(df_nepal):,} of {len(df):,} rows.")

Kept 1,422 of 1,579 rows.


In [22]:
# Data distribution for the filtered earthquakes
df_nepal.describe()

,latitude,longitude,depth,mag,nst,gap,dmin,rms,horizontalError,depthError,magError,magNst
count,1422.000000,1422.00000,1422.000000,1422.000000,530.000000,954.000000,718.000000,1334.000000,705.000000,941.000000,744.000000,1286.000000
mean,28.745938,85.14298,23.108175,4.459437,48.600000,112.500419,2.629169,0.889168,8.001986,4.966465,0.112946,31.713841
std,0.987891,2.41675,17.055521,0.496592,50.111354,48.312816,2.346038,0.277748,2.449872,5.939411,0.061212,41.889411
min,26.517000,80.09800,2.000000,3.000000,5.000000,10.000000,0.027000,0.100000,2.400000,0.000000,0.033000,1.000000
25%,27.841400,83.38300,10.000000,4.100000,18.000000,76.000000,0.799750,0.690000,6.300000,1.889000,0.074750,6.000000
50%,28.590200,85.81880,15.000000,4.400000,33.000000,107.000000,1.981500,0.870000,7.800000,1.901000,0.097000,17.000000
75%,29.646300,87.27320,33.000000,4.700000,61.000000,152.000000,3.683250,1.090000,9.500000,5.600000,0.141000,39.750000
max,30.448000,88.20000,143.200000,7.800000,383.000000,290.500000,22.604000,1.890000,24.720000,45.800000,0.580000,328.000000


In [23]:
df['time'].dtype

dtype('O')

In [31]:
# Split the UTC timestamp so we can group earthquakes by date or time of day.
dt = pd.to_datetime(df.loc[df_nepal.index, "time"], utc=True).dt

df_nepal["year"] = dt.year
df_nepal["month"] = dt.month
df_nepal["day"] = dt.day
df_nepal["time_of_day"] = dt.time
df_nepal = df_nepal.drop(columns="time", errors="ignore")

df_nepal.head()

,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,net,...,depthError,magError,magNst,status,locationSource,magSource,year,month,day,time_of_day
0,27.8692,87.8857,10.0,4.6,mb,40.0,161.0,9.216,0.91,us,...,1.910,0.093,34.0,reviewed,us,us,2026,11,11,23:00:22.433000
1,29.6017,80.8002,10.0,4.1,mb,22.0,166.0,7.584,0.99,us,...,1.890,0.185,8.0,reviewed,us,us,2026,10,25,07:21:37.857000
3,28.0314,87.3256,10.0,4.5,mb,55.0,159.0,9.685,0.65,us,...,1.887,0.081,44.0,reviewed,us,us,2026,10,22,17:11:27.957000
4,28.6874,87.3652,10.0,4.2,mb,19.0,160.0,10.117,0.71,us,...,1.920,0.166,11.0,reviewed,us,us,2026,10,16,19:35:17.741000
6,29.6348,81.7592,10.0,4.0,mb,18.0,117.0,8.270,0.48,us,...,1.984,0.231,5.0,reviewed,us,us,2026,9,15,22:43:11.450000


In [36]:
# Save cleaned earthquake data
output_path = data_path.parent.parent / "processed" / "earthquakes.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

df_nepal.to_csv(output_path, index=False)

print(f"Saved {len(df_nepal):,} earthquakes to {output_path}")

Saved 1,422 earthquakes to ../data/processed/earthquakes.csv
